In [ ]:
from pathlib import Path
import json, random, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)


In [ ]:
PROJECT_ROOT=Path("..")
DATA_DIR=PROJECT_ROOT/"data"/"processed"
RESULTS_DIR=PROJECT_ROOT/"outputs"/"results"
FIGURES_DIR=PROJECT_ROOT/"outputs"/"figures"
MODELS_DIR=PROJECT_ROOT/"saved-models"
for p in (RESULTS_DIR,FIGURES_DIR,MODELS_DIR): p.mkdir(parents=True,exist_ok=True)

# Change only this filename if Member 1 uses another name.
DATA_PATH=DATA_DIR/"final_cicids2017.parquet"
print(DATA_PATH.resolve())


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Update DATA_PATH when Member 1 gives you the final file.")
data = pd.read_parquet(DATA_PATH) if DATA_PATH.suffix.lower()==".parquet" else pd.read_csv(DATA_PATH,low_memory=False)
if "Label" not in data.columns: raise KeyError("Required column 'Label' is missing.")
print("Shape:",data.shape)
display(data["Label"].value_counts().to_frame("Count"))


In [ ]:
UNKNOWN_ATTACKS=["Bot","Infiltration","Web Attack - SQL Injection","Heartbleed"]
normal_data=data.loc[data["Label"]=="BENIGN"].copy()
unknown_data=data.loc[data["Label"].isin(UNKNOWN_ATTACKS)].copy()
if normal_data.empty or unknown_data.empty: raise ValueError("Required BENIGN/UNKNOWN records not found.")
print("NORMAL:",len(normal_data))
print("UNKNOWN:",len(unknown_data))
display(unknown_data["Label"].value_counts().to_frame("Count"))


In [ ]:
# Reserve an equal number of BENIGN flows for the final balanced test.
n_unknown=len(unknown_data)
final_normal=normal_data.sample(n=n_unknown,random_state=SEED)
normal_dev=normal_data.drop(index=final_normal.index)

# Autoencoder development data is BENIGN only.
normal_train,normal_val=train_test_split(normal_dev,test_size=0.10,random_state=SEED,shuffle=True)
print("Train:",len(normal_train),"Validation:",len(normal_val),"Final NORMAL:",len(final_normal),"Final UNKNOWN:",len(unknown_data))


In [ ]:
# Member 1 has already cleaned/selected features. Only target/metadata are excluded here.
NON_FEATURE_COLUMNS=["Label","source_file"]
features=[c for c in data.columns if c not in NON_FEATURE_COLUMNS]
bad=[c for c in features if not pd.api.types.is_numeric_dtype(data[c])]
if bad: raise TypeError(f"Non-numeric model features remain: {bad}")

X_train=normal_train[features].copy()
X_val=normal_val[features].copy()
X_test_normal=final_normal[features].copy()
X_test_unknown=unknown_data[features].copy()

for name,df in {"train":X_train,"val":X_val,"normal_test":X_test_normal,"unknown_test":X_test_unknown}.items():
    nans=int(df.isna().sum().sum()); infs=int(np.isinf(df.to_numpy(dtype=np.float64)).sum())
    print(name,df.shape,"NaN:",nans,"Inf:",infs)
    if nans or infs: raise ValueError("Final dataset still contains NaN/Inf. Resolve with Member 1.")


In [ ]:
# Set True only if Member 1 explicitly provides already-standardized features.
ALREADY_SCALED=False
if ALREADY_SCALED:
    scaler=None
    X_train_s=X_train.to_numpy(np.float32); X_val_s=X_val.to_numpy(np.float32)
    X_test_normal_s=X_test_normal.to_numpy(np.float32); X_test_unknown_s=X_test_unknown.to_numpy(np.float32)
else:
    scaler=StandardScaler()
    X_train_s=scaler.fit_transform(X_train).astype(np.float32)
    X_val_s=scaler.transform(X_val).astype(np.float32)
    X_test_normal_s=scaler.transform(X_test_normal).astype(np.float32)
    X_test_unknown_s=scaler.transform(X_test_unknown).astype(np.float32)


In [ ]:
input_dim=X_train_s.shape[1]
inputs=Input(shape=(input_dim,),name="network_flow")
x=Dense(48,activation="relu")(inputs)
x=Dense(24,activation="relu")(x)
latent=Dense(12,activation="relu",name="latent_space")(x)
x=Dense(24,activation="relu")(latent)
x=Dense(48,activation="relu")(x)
outputs=Dense(input_dim,activation="linear")(x)
autoencoder=Model(inputs,outputs,name="unknown_attack_autoencoder")
autoencoder.compile(optimizer="adam",loss="mse")
autoencoder.summary()


In [ ]:
early=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)
history=autoencoder.fit(
    X_train_s,X_train_s,
    validation_data=(X_val_s,X_val_s),
    epochs=50,batch_size=1024,shuffle=True,
    callbacks=[early],verbose=1
)


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"],label="Training loss")
plt.plot(history.history["val_loss"],label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("MSE loss"); plt.title("Autoencoder training history"); plt.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR/"member3_autoencoder_training_history.png",dpi=150)
plt.show()


In [ ]:
val_rec=autoencoder.predict(X_val_s,batch_size=4096,verbose=0)
val_error=np.mean(np.square(X_val_s-val_rec),axis=1)
THRESHOLD_PERCENTILE=99.0
threshold=float(np.percentile(val_error,THRESHOLD_PERCENTILE))
print("Validation mean error:",val_error.mean())
print("Threshold:",threshold)


In [ ]:
# Final test: 50% NORMAL + 50% held-out UNKNOWN.
X_test=np.vstack([X_test_normal_s,X_test_unknown_s])
y_test=np.concatenate([np.zeros(len(X_test_normal_s),dtype=np.int8),np.ones(len(X_test_unknown_s),dtype=np.int8)])

test_rec=autoencoder.predict(X_test,batch_size=4096,verbose=0)
test_error=np.mean(np.square(X_test-test_rec),axis=1)
y_pred=(test_error>threshold).astype(np.int8)

cm=confusion_matrix(y_test,y_pred); TN,FP,FN,TP=cm.ravel()
metrics=pd.DataFrame([{
    "accuracy":accuracy_score(y_test,y_pred),
    "precision":precision_score(y_test,y_pred,zero_division=0),
    "unknown_detection_rate":recall_score(y_test,y_pred,zero_division=0),
    "f1_score":f1_score(y_test,y_pred,zero_division=0),
    "roc_auc":roc_auc_score(y_test,test_error),
    "false_positive_rate":FP/(FP+TN),
    "TN":TN,"FP":FP,"FN":FN,"TP":TP,
    "threshold":threshold
}])
display(metrics)
print(classification_report(y_test,y_pred,target_names=["NORMAL","UNKNOWN"],digits=4,zero_division=0))


In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.xticks([0,1],["NORMAL","UNKNOWN"]); plt.yticks([0,1],["NORMAL","UNKNOWN"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Final NORMAL vs UNKNOWN confusion matrix")
for i in range(2):
    for j in range(2): plt.text(j,i,str(cm[i,j]),ha="center",va="center")
plt.tight_layout(); plt.savefig(FIGURES_DIR/"member3_final_confusion_matrix.png",dpi=150); plt.show()


In [ ]:
# Research-only breakdown: original labels were never model inputs.
start=len(X_test_normal_s)
unknown_analysis=pd.DataFrame({
    "original_attack":unknown_data["Label"].to_numpy(),
    "reconstruction_error":test_error[start:],
    "detected_as_anomaly":y_pred[start:]
})
per_attack=unknown_analysis.groupby("original_attack").agg(
    samples=("detected_as_anomaly","size"),
    detected=("detected_as_anomaly","sum"),
    mean_reconstruction_error=("reconstruction_error","mean")
)
per_attack["detection_rate"]=per_attack["detected"]/per_attack["samples"]
display(per_attack)


In [ ]:
metrics.to_csv(RESULTS_DIR/"member3_final_metrics.csv",index=False)
per_attack.to_csv(RESULTS_DIR/"member3_per_attack_analysis.csv")
pd.DataFrame({
    "subset":["NORMAL train","NORMAL validation","NORMAL final test","UNKNOWN final test"],
    "samples":[len(normal_train),len(normal_val),len(final_normal),len(unknown_data)]
}).to_csv(RESULTS_DIR/"member3_data_split_summary.csv",index=False)

autoencoder.save(MODELS_DIR/"member3_unknown_autoencoder.keras")
if scaler is not None: joblib.dump(scaler,MODELS_DIR/"member3_standard_scaler.joblib")
with open(MODELS_DIR/"member3_threshold.json","w") as f: json.dump({"percentile":THRESHOLD_PERCENTILE,"threshold":threshold},f,indent=2)
with open(MODELS_DIR/"member3_feature_columns.json","w") as f: json.dump(features,f,indent=2)
print("All Member 3 outputs saved.")


## Final output
The autoencoder returns a reconstruction error. If it is above the validation-derived threshold, the flow is flagged as **ANOMALY / POTENTIAL UNKNOWN ATTACK**; otherwise it is **NORMAL**.
